# 🚀 HUẤN LUYỆN MÔ HÌNH UNIFIED ReID (NGƯỜI & XE) CHO UKF-DeepSORT

**Mục tiêu hệ thống:**
- **Kiến trúc:** MobileNetV4-Conv-Small (Google 2024, ~2.5M params) + Projection Head (256D) + BNNeck.
- **Đầu vào:** Chuẩn hóa hình vuông $128 \times 128$ (Letterbox Padding) chung cho cả Người & Xe.
- **Đầu ra:** Vector đặc trưng 256 chiều đã chuẩn hóa L2 (L2-Normalized Embedding).
- **Huấn luyện siêu tốc:** GPU T4 (Mixed Precision FP16 AMP) chỉ mất **~35 - 45 phút**.
- **Xuất xưởng:** Tự động lượng tử hóa sang **OpenVINO INT8 (NNCF)** với dung lượng ~2.5 MB, tải thẳng về máy.

👉 **LƯU Ý QUAN TRỌNG:** Vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** trước khi chạy!

In [ ]:
# CELL 1: Cài đặt thư viện (Cố định protobuf tránh xung đột môi trường Colab)
!pip install -q "protobuf>=5.29.1,<6.0.0dev" timm openvino nncf kaggle onnx torchvision tqdm gdown

import os
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU đang sử dụng: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CẢNH BÁO: Bạn chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU")

## CELL 2: Tải Dữ liệu Huấn luyện (Google Drive & Kaggle)
- **Market-1501 (Người đi bộ):** Tải trực tiếp từ Google Drive của nhóm tác giả Liang Zheng qua `gdown` (Không bao giờ bị lỗi 403).
- **VeRi-776 (Xe hơi):** Tải qua Kaggle CLI với dataset công khai của `ahmetfurkandemir`.

In [ ]:
import os
import shutil
from glob import glob

os.makedirs("data/market1501", exist_ok=True)
os.makedirs("data/veri776", exist_ok=True)

# 1. TẢI TẬP DỮ LIỆU NGƯỜI (Market-1501) TỪ GOOGLE DRIVE CHÍNH THỨC
print("📥 Đang tải tập dữ liệu Market-1501 từ Google Drive (Tốc độ nội bộ ~50-80 MB/s)...")
market_train_dir = "data/market1501/Market-1501-v15.09.15/bounding_box_train"
if not os.path.exists(market_train_dir):
    !gdown --id 0B8-rUzbwVRk0c054eEozWG9COHM -O market1501.zip
    !unzip -q market1501.zip -d data/market1501
    if os.path.exists("market1501.zip"): os.remove("market1501.zip")

market_count = len(glob(f"{market_train_dir}/*.jpg"))
print(f"✅ Đã sẵn sàng Market-1501: {market_count} ảnh người.")

# 2. TẢI TẬP DỮ LIỆU XE (VeRi-776 / Hugging Face Car Dataset)
print("\n📥 Đang chuẩn bị tập dữ liệu Xe hơi...")
veri_path = None

# Cách 1: Thử tải VeRi-776 qua kagglehub
try:
    import kagglehub
    veri_path = kagglehub.dataset_download("abhyudaya12/veri-vehicle-reidentification-dataset")
    print(f"✅ Tải thành công VeRi-776 qua Kaggle tại: {veri_path}")
except Exception as e:
    print(f"Kaggle yêu cầu xác thực hoặc gặp lỗi: {e}")
    print("🚀 Đang tự động chuyển sang tải tập ảnh Xe chất lượng cao từ Hugging Face (Không cần tài khoản, tải tức thì)...\n")

# Cách 2 (Cứu cánh 100% không bao giờ lỗi 403): Tải tập Xe từ Hugging Face
if not veri_path or not os.path.exists(veri_path):
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    veri_path = snapshot_download(repo_id="HumynLabs/car-images", repo_type="dataset")
    print(f"✅ Đã tải thành công tập dữ liệu Xe từ Hugging Face: {veri_path}")

# Kiểm tra số lượng ảnh xe
veri_imgs = (glob(f"{veri_path}/**/*.jpg", recursive=True) + glob(f"{veri_path}/**/*.png", recursive=True)) if veri_path else []
print(f"✅ Đã sẵn sàng tập Xe: {len(veri_imgs)} ảnh xe hơi.")

## CELL 3: Xử lý Dữ liệu & Balanced P-K Sampler
- Gộp ID người và ID xe vào chung một bảng nhãn toàn cục (`global_pid`).
- **Balanced P-K Sampler:** Mỗi batch 64 ảnh luôn chia đều: **8 người + 8 xe** (mỗi danh tính lấy 4 ảnh).

In [ ]:
import re
import random
from collections import defaultdict
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

IMG_SIZE = (128, 128)

# Data Augmentation tối ưu cho ReID (Random Erasing chống che khuất)
train_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10, padding_mode='reflect'),
    T.RandomCrop(IMG_SIZE),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value='random')
])

samples = []  # (image_path, global_pid, domain_tag)
global_id_map = {}
current_id = 0

# 1. Quét tập Người (Market-1501)
market_files = glob("data/market1501/**/bounding_box_train/*.jpg", recursive=True)
for p in market_files:
    fname = os.path.basename(p)
    if fname.startswith('-1') or fname.startswith('0000'): continue
    raw_id = fname.split('_')[0]
    key = f"person_{raw_id}"
    if key not in global_id_map:
        global_id_map[key] = current_id
        current_id += 1
    samples.append((p, global_id_map[key], 'person'))

# 2. Quét tập Xe (VeRi / Hugging Face Cars)
veri_files = (glob(f"{veri_path}/**/*.jpg", recursive=True) + glob(f"{veri_path}/**/*.png", recursive=True)) if veri_path else []
veri_files = veri_files[:20000]
for p in veri_files:
    fname = os.path.basename(p)
    raw_id = fname.split('_')[0] if '_' in fname else os.path.basename(os.path.dirname(p))
    key = f"vehicle_{raw_id}"
    if key not in global_id_map:
        global_id_map[key] = current_id
        current_id += 1
    samples.append((p, global_id_map[key], 'vehicle'))

num_classes = current_id
print(f"📊 Tổng cộng: {len(samples)} ảnh huấn luyện thuộc {num_classes} danh tính (Người + Xe).")

class UnifiedReIDDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        self.pid_dict = defaultdict(list)
        for idx, (path, pid, tag) in enumerate(self.samples):
            self.pid_dict[pid].append(idx)
        self.pids = list(self.pid_dict.keys())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, pid, tag = self.samples[index]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, pid, tag

class BalancedPKIdentitySampler(Sampler):
    def __init__(self, dataset, p_per_class=8, k_samples=4):
        self.dataset = dataset
        self.p_per_class = p_per_class
        self.k_samples = k_samples
        self.person_pids = [pid for pid in dataset.pids if dataset.samples[dataset.pid_dict[pid][0]][2] == 'person']
        self.vehicle_pids = [pid for pid in dataset.pids if dataset.samples[dataset.pid_dict[pid][0]][2] == 'vehicle']

    def __iter__(self):
        person_pids = self.person_pids.copy()
        vehicle_pids = self.vehicle_pids.copy()
        random.shuffle(person_pids)
        random.shuffle(vehicle_pids)
        batch = []
        min_len = min(len(person_pids), len(vehicle_pids))
        for i in range(0, min_len - self.p_per_class, self.p_per_class):
            selected_p = person_pids[i:i+self.p_per_class] + vehicle_pids[i:i+self.p_per_class]
            for pid in selected_p:
                t = self.dataset.pid_dict[pid]
                if len(t) >= self.k_samples:
                    t = random.sample(t, self.k_samples)
                else:
                    t = [random.choice(t) for _ in range(self.k_samples)]
                batch.extend(t)
                if len(batch) == (self.p_per_class * 2 * self.k_samples):
                    yield from batch
                    batch = []

    def __len__(self):
        return min(len(self.person_pids), len(self.vehicle_pids)) * 2 * self.k_samples

dataset = UnifiedReIDDataset(samples, transform=train_transform)
sampler = BalancedPKIdentitySampler(dataset, p_per_class=8, k_samples=4)
train_loader = DataLoader(dataset, batch_size=64, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)
print("✅ Khởi tạo DataLoader thành công! Batch size = 64.")

## CELL 4: Định nghĩa Mô hình MobileNetV4-Conv-Small + BNNeck
- Backbone: `mobilenetv4_conv_small` (pre-trained ImageNet, tối ưu siêu tốc cho OpenVINO CPU).
- Head: Linear 1280D -> 256D -> BNNeck -> L2 Normalization.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import timm

class UnifiedMobileNetV4ReID(nn.Module):
    def __init__(self, num_classes, embedding_dim=256):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv4_conv_small.e1200_r224_in1k', pretrained=True, num_classes=0)
        # Tự động lấy kích thước đầu ra thực tế của backbone (1280D)
        with torch.no_grad():
            dummy_out = self.backbone(torch.zeros(1, 3, 128, 128))
            in_features = dummy_out.shape[1]
        
        self.fc = nn.Linear(in_features, embedding_dim, bias=False)
        self.bottleneck = nn.BatchNorm1d(embedding_dim)
        self.bottleneck.bias.requires_grad_(False)
        self.classifier = nn.Linear(embedding_dim, num_classes, bias=False)
        self.embedding_dim = embedding_dim

    def forward(self, x):
        feat = self.backbone(x)
        feat_proj = self.fc(feat)
        feat_bn = self.bottleneck(feat_proj)
        
        if not self.training:
            # Trả về vector 256D chuẩn hóa L2 cho Tracker
            return F.normalize(feat_bn, p=2, dim=1)
            
        cls_score = self.classifier(feat_bn)
        return cls_score, feat_proj

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UnifiedMobileNetV4ReID(num_classes=num_classes, embedding_dim=256).to(device)
print(f"✅ Mô hình đã sẵn sàng trên thiết bị: {device}")
print(f"📊 Số tham số: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params.")

## CELL 5: Hàm Mất Mát (Loss Functions) & Bộ Tối Ưu

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=-1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets = (1 - self.epsilon) * targets + self.epsilon / self.num_classes
        loss = (-targets * log_probs).mean(0).sum()
        return loss

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        # Ép kiểu inputs về float32 để tính khoảng cách chính xác, tránh lệch dtype với autocast FP16
        inputs = inputs.float()
        n = inputs.size(0)
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist = dist.addmm(inputs, inputs.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()
        
        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []
        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))
        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        
        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)

criterion_ce = CrossEntropyLabelSmooth(num_classes=num_classes)
criterion_tri = BatchHardTripletLoss(margin=0.3)

optimizer = torch.optim.AdamW(model.parameters(), lr=3.5e-4, weight_decay=5e-4)
NUM_EPOCHS = 60
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()  # Tăng tốc Mixed Precision FP16
print("✅ Cấu hình Loss, Optimizer & AMP Scaler hoàn tất!")

## CELL 6: Huấn luyện Mô hình với PyTorch AMP trên GPU T4 (~40 phút)

In [ ]:
import time
from tqdm import tqdm

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN {NUM_EPOCHS} EPOCHS TRÊN GPU T4...")
model.train()
start_train_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    epoch_start = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS}")
    
    for imgs, pids, tags in pbar:
        imgs, pids = imgs.to(device), pids.to(device)
        optimizer.zero_grad()
        
        # Tăng tốc độ với PyTorch Autocast FP16
        with torch.cuda.amp.autocast():
            cls_score, feat_proj = model(imgs)
            loss_ce = criterion_ce(cls_score, pids)
            loss_tri = criterion_tri(feat_proj, pids)
            loss = loss_ce + loss_tri
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.3f}", "LR": f"{scheduler.get_last_lr()[0]:.6f}"})
        
    scheduler.step()
    elapsed = time.time() - epoch_start
    avg_loss = epoch_loss / max(len(train_loader), 1)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} - Avg Loss: {avg_loss:.4f} - Time: {elapsed:.1f}s")
    
print(f"\n🎉 HUẤN LUYỆN HOÀN TẤT! Tổng thời gian: {(time.time() - start_train_time) / 60:.1f} phút.")
torch.save(model.state_dict(), "unified_reid_best.pth")
print("💾 Đã lưu checkpoint: unified_reid_best.pth")

## CELL 7: Tự Động Lượng Tử Hóa OpenVINO INT8 (NNCF) & Đóng Gói File
Xuất mô hình sang định dạng OpenVINO INT8 với kích thước siêu nhẹ (~2.5 MB), tối ưu tập lệnh VNNI trên CPU Intel.

In [ ]:
import openvino as ov
import nncf

save_dir = "unified_openvino_export"
os.makedirs(save_dir, exist_ok=True)
model.eval()
model.cpu()

# 1. Xuất sang ONNX
dummy_input = torch.randn(1, 3, 128, 128)
onnx_path = os.path.join(save_dir, "unified_reid.onnx")
print("1. Đang xuất đồ thị mô hình sang ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=14
)

# 2. Chuyển đổi sang OpenVINO Model (Static 1x3x128x128 cho CPU throughut)
print("2. Chuyển đổi sang OpenVINO Model...")
core = ov.Core()
ov_model = core.read_model(onnx_path)
ov_model.reshape([1, 3, 128, 128])

fp16_path = os.path.join(save_dir, "unified_reid_fp16.xml")
ov.save_model(ov_model, fp16_path)
print(f"✅ Đã lưu bản OpenVINO FP16 tại: {fp16_path}")

# 3. Lượng tử hóa NNCF INT8 bằng tập hiệu chuẩn (Calibration)
print("3. Đang thực hiện lượng tử hóa Post-Training INT8 (NNCF)...")
calibration_samples = []
for idx in range(min(300, len(dataset))):
    img_tensor, _, _ = dataset[idx]
    calibration_samples.append(img_tensor.unsqueeze(0).numpy())

calib_dataset = nncf.Dataset(calibration_samples)
quantized_model = nncf.quantize(ov_model, calib_dataset)

int8_path = os.path.join(save_dir, "unified_reid_int8.xml")
ov.save_model(quantized_model, int8_path)
print(f"✅ ĐÃ TẠO THÀNH CÔNG MÔ HÌNH INT8: {int8_path}")

# 4. Đóng gói file ZIP
!zip -j unified_reid_openvino_int8.zip unified_openvino_export/*
print("\n📦 ĐÃ ĐÓNG GÓI THÀNH CÔNG FILE: unified_reid_openvino_int8.zip")

## CELL 8: Tải file về máy tính

In [ ]:
from google.colab import files
print("📥 Đang tải file unified_reid_openvino_int8.zip về máy tính của bạn...")
try:
    files.download("unified_reid_openvino_int8.zip")
except Exception as e:
    print("Nếu file không tự động tải, hãy nhìn vào cây thư mục bên trái của Colab và bấm chuột phải vào 'unified_reid_openvino_int8.zip' chọn Download.")